# unbox-args-tensor-to-array — ex2: unbox_args_nested: recurse into list/tuple args, preserve container

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbox-args-tensor-to-array`. Running the final beacon cell reports progress against the `Backprop: Unbox Tensor args to array` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbox Tensor args to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbox-args-tensor-to-array`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbox-args-tensor-to-array"
DD_SUBTOPIC = "Backprop: Unbox Tensor args to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Unbox — handling nested args (lists/tuples of Tensors) — quick refresher

Some forward ops take a SEQUENCE of tensors as a single positional arg:

```python
stacked = cat([t1, t2, t3], dim=0)        # arg 0 is a list of MiniTensors
result  = stack((t_a, t_b), dim=1)         # arg 0 is a tuple of MiniTensors
```

The raw `torch.cat` accepts `list[Tensor]`, not `list[MiniTensor]`. So `unbox_args` must RECURSE into list/tuple args and unbox the inner Tensors, while preserving the container type (list stays list, tuple stays tuple).

Critical: do NOT recurse forever — only one level of nesting is the canonical pattern in PyTorch (no `cat([[t1, t2]], dim=0)`).

### Exercise 2 — unbox_args_nested: recurse into list/tuple args, preserve container

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply unboxing to nested args: when a positional arg is a list or tuple of MiniTensors, recurse one level and preserve the container type so `cat`/`stack`-style ops work.
> Keywords: unbox, nested, cat, stack, container-type
> ```

**KCs targeted:** `unbox-args-tensor-to-array`, `parents-dict-by-argidx`

Implement `unbox_args_nested(args)`. Same shape as ex1's `unbox_args`, BUT: if a positional arg is a `list` or `tuple` whose elements include `MiniTensor` instances, recurse one level — replace each inner `MiniTensor` with its `.array`, preserving the container type:

```
unbox_args_nested(([t1, t2, t3], 0))    == ([t1.array, t2.array, t3.array], 0)
unbox_args_nested(((tA, tB), 1))         == ((tA.array, tB.array), 1)
unbox_args_nested((t1, 3.0, t2))         == (t1.array, 3.0, t2.array)  # ex1 case
unbox_args_nested(([1, 2, 3],))          == ([1, 2, 3],)               # plain ints stay
```

Rules:

**1. Single level of recursion.** A list-of-list is preserved as-is (no PyTorch op uses that signature). Don't go infinitely deep — that risks pathological inputs.

**2. Preserve container type.** `list` in → `list` out. `tuple` in → `tuple` out. Don't normalize to one.

**3. Mixed inner types are fine.** `[t1, 3.0, t2]` → `[t1.array, 3.0, t2.array]` — inner pass-through, same as the top level.

**4. Top-level `MiniTensor` still unboxes** (ex1 behavior — this function generalizes ex1, doesn't replace it).

Why this matters: `cat`/`stack` are the canonical example. Without nested unbox, `cat([m1, m2])` reaches the raw `torch.cat` with a `list[MiniTensor]`, which crashes (`AttributeError: ... has no attribute 'dim'`).

In [ ]:
def unbox_args_nested(args: tuple) -> tuple:
    def _maybe_unbox(a):
        if isinstance(a, MiniTensor):
            return a.array
        if isinstance(a, list):
            return [_maybe_unbox(x) for x in a]
        if isinstance(a, tuple):
            return tuple(_maybe_unbox(x) for x in a)
        return a
    return tuple(_maybe_unbox(a) for a in args)


<details><summary>Solution</summary>

```python
def unbox_args_nested(args: tuple) -> tuple:
    def _maybe_unbox(a):
        if isinstance(a, MiniTensor):
            return a.array
        if isinstance(a, list):
            return [_maybe_unbox(x) for x in a]
        if isinstance(a, tuple):
            return tuple(_maybe_unbox(x) for x in a)
        return a
    return tuple(_maybe_unbox(a) for a in args)
```

**One level of recursion is intentional.** A `list[list[Tensor]]` is not a real PyTorch op signature. Recursing arbitrarily deep risks pathological inputs (cycles, ragged structures) for no real-world payoff. The inner helper happens to be recursive but the input set never goes more than 2 deep in practice.

**Container-type preservation matters.** `torch.cat` accepts list and tuple interchangeably, but `torch.stack` historically wants the same type back. Normalizing to one type can break the raw fn.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()